In [ ]:
!pip install pandas scikit-learn google-cloud-storage joblib

In [1]:
!git clone --branch week_1 --single-branch https://github.com/IITMBSMLOps/ga_resources.git

Cloning into 'ga_resources'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 10.07 KiB | 793.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.


In [3]:
import os

for root, dirs, files in os.walk("ga_resources/data"):
    for file in files:
        print(os.path.join(root, file))

ga_resources/data/v2/data.csv
ga_resources/data/raw/iris.csv
ga_resources/data/v1/data.csv


In [6]:
### SPLIT FUNCTION ###
import pandas as pd
from sklearn.model_selection import train_test_split
def create_split(
    input_file,
    version
):
    df = pd.read_csv(
        input_file
    )
    target = df.columns[-1]
    train_df, eval_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df[target]
    )
    train_df.to_csv(
        f"{version}_train.csv",
        index=False
    )
    eval_df.to_csv(
        f"{version}_eval.csv",
        index=False
    )
    print(f"{version} complete")

In [8]:
### CREATE SPLIT ###
create_split(
    "ga_resources/data/raw/iris.csv",
    "raw"
)

create_split(
    "ga_resources/data/v1/data.csv",
    "v1"
)

create_split(
    "ga_resources/data/v2/data.csv",
    "v2"
)

raw complete
v1 complete
v2 complete


In [19]:
### UPLOAD FILES ###
from google.cloud import storage
from pathlib import Path
client = storage.Client()
bucket = client.bucket(
    "irisdata-week1"
)
for file in [
    "ga_resources/data/v2/data.csv",
    "ga_resources/data/raw/iris.csv",
    "ga_resources/data/v1/data.csv"
]: 
    path = Path(file)
    parent_folder = path.parent.name
    filename = path.name
    gcs_filename = f"{parent_folder}_{filename}"
    bucket.blob(
        f"raw/{gcs_filename}"
    ).upload_from_filename(file)
    
for file in [
    "raw_train.csv",
    "v1_train.csv",
    "v2_train.csv"
]:
    bucket.blob(
        f"train/{file}"
    ).upload_from_filename(
        file
    )

for file in [
    "raw_eval.csv",
    "v1_eval.csv",
    "v2_eval.csv"
]:
    bucket.blob(
        f"eval/{file}"
    ).upload_from_filename(
        file
    )

In [20]:
### FIRST TRAIN RUN ###
!python train.py --data_version raw

Downloaded train/raw_train.csv

Training Complete
Accuracy: 1.0
Artifact Folder: raw_20260617_170058


In [21]:
### FIRST INFER RUN ###
!python inference.py \
  --data_version raw \
  --artifact_folder raw_20260617_170058


Inference Complete
Accuracy: 0.9
Artifact Folder: raw_20260617_170058


In [22]:
### SECOND TRAIN RUN ###
!python train.py --data_version raw

Downloaded train/raw_train.csv

Training Complete
Accuracy: 1.0
Artifact Folder: raw_20260617_170131


In [23]:
### SECOND INFER RUN ###
!python inference.py \
  --data_version raw \
  --artifact_folder raw_20260617_170131


Inference Complete
Accuracy: 0.9
Artifact Folder: raw_20260617_170131


In [24]:
### V1 TRAIN RUN ###
!python train.py --data_version v1

Downloaded train/v1_train.csv

Training Complete
Accuracy: 1.0
Artifact Folder: v1_20260617_170149


In [25]:
### V1 INFER RUN ###
!python inference.py \
  --data_version v1 \
  --artifact_folder v1_20260617_170149


Inference Complete
Accuracy: 0.9523809523809523
Artifact Folder: v1_20260617_170149


In [26]:
### V2 TRAIN RUN ###
!python train.py --data_version v2

Downloaded train/v2_train.csv

Training Complete
Accuracy: 1.0
Artifact Folder: v2_20260617_170204


In [27]:
### V2 INFER RUN ###
!python inference.py \
  --data_version v2 \
  --artifact_folder v2_20260617_170204


Inference Complete
Accuracy: 1.0
Artifact Folder: v2_20260617_170204


In [29]:
from google.cloud import storage
import pandas as pd
import json

ARTIFACT_BUCKET = "irisdata-week1-artefacts"
client = storage.Client()
bucket = client.bucket(ARTIFACT_BUCKET)
rows = []

# Find all inference metrics files
for blob in client.list_blobs(
    ARTIFACT_BUCKET
):
    if blob.name.endswith(
        "inference_metrics.json"
    ):
        content = blob.download_as_text()
        metrics = json.loads(content)
        rows.append(
            {
                "artifact_folder": metrics["artifact_folder"],
                "data_version": metrics["data_version"],
                "accuracy": metrics["accuracy"]
            }
        )
comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values(
    by="data_version"
)
comparison_df

,artifact_folder,data_version,accuracy
0,raw_20260617_170058,raw,0.900000
1,raw_20260617_170131,raw,0.900000
2,v1_20260617_170149,v1,0.952381
3,v2_20260617_170204,v2,1.000000


In [30]:
comparison_df.to_csv(
    "comparison.csv",
    index=False
)

comparison_df

,artifact_folder,data_version,accuracy
0,raw_20260617_170058,raw,0.900000
1,raw_20260617_170131,raw,0.900000
2,v1_20260617_170149,v1,0.952381
3,v2_20260617_170204,v2,1.000000
